# 05 — Combine H→WW and H→bb Fits

Build combined chi2 on a shared WC grid: `chi2_total = chi2_hww + chi2_hbb`.

In [ ]:
from pathlib import Path
import json, sys, os, math
import numpy as np
import pandas as pd

def find_repo_root(start=None, marker='smeft_contents.txt', max_up=8):
    p = Path(start or Path.cwd()).resolve()
    for _ in range(max_up+1):
        if (p / marker).exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(f'Could not find repo root containing {marker} from {Path.cwd()}')

REPO = find_repo_root()
NOTEBOOK_DIR = REPO / 'notebooks_hww_fit'
SMEFT_LISTING = REPO / 'smeft_contents.txt'
LOCAL_DICT = NOTEBOOK_DIR / 'dictionary.json'
print('repo:', REPO)
import importlib
import matplotlib.pyplot as plt
P_ANALYSIS=Path('/uscms_data/d3/azhou/smeft/analysis')
HBB=P_ANALYSIS/'hbb-coffea'
for x in [P_ANALYSIS,HBB,NOTEBOOK_DIR]:
    if str(x) not in sys.path: sys.path.append(str(x))
import stxs_functions as sf
importlib.reload(sf)

In [ ]:
def combine_1d(df_hww, df_hbb, wc_col='wc'):
    a=df_hww[[wc_col,'chi2_hww']].copy()
    b=df_hbb[[wc_col,'chi2_total']].copy().rename(columns={'chi2_total':'chi2_hbb'})
    m=a.merge(b,on=wc_col,how='inner')
    m['chi2_comb']=m['chi2_hww']+m['chi2_hbb']
    return m.sort_values('chi2_comb')

def one_sigma_interval(df, wc_col='wc', chi2_col='chi2_comb', delta=1.0):
    cmin=df[chi2_col].min()
    allowed=df[df[chi2_col] <= cmin+delta]
    return allowed[wc_col].min(), allowed[wc_col].max(), cmin

In [ ]:
# Example workflow:
# from 03 notebook: df_hww with columns wc,chi2_hww
# from 04 notebook: df_hbb with columns wc,chi2_total
# comb = combine_1d(df_hww, df_hbb)
# lo,hi,cmin = one_sigma_interval(comb)
# print('combined 1sigma:',(lo,hi),'chi2min=',cmin)